In [1]:
import sys
from pathlib import Path

# Move up one level from 'notebooks' to the root directory and add it to sys.path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

In [24]:
from functools import partial
from torch.optim import Adam
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random
import time
from pathlib import Path

from src.preprocessing.text import build_vocabulary
from src.data.loader import LABEL_COLUMNS
from src.data.split import split_train_validation
from src.deep_learning.model import ToxicCommentBiLSTM
from src.deep_learning.trainer import (
    train_one_epoch,
    evaluate,
    predict,
)
from src.deep_learning.dataset import (
    ToxicCommentDataset,
    collate_fn,
)
from src.evaluation.metrics import calculate_metrics

EPOCHS = 5
LEARNING_RATE = 0.001
MAX_POS_WEIGHT = 20.0
SEED = 42

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

Device: cuda


In [14]:
# 1. Split the dataset.
train_df, val_df = split_train_validation()

# 2. Build vocabulary using training comments only.
vocab = build_vocabulary(
    train_df["comment_text"],
    min_freq=2,
    max_vocab_size=30_000,
)

# 3. Create training dataset.
train_dataset = ToxicCommentDataset(
    train_df,
    vocab,
    max_length=300,
)

# 4. Create validation dataset.
val_dataset = ToxicCommentDataset(
    val_df,
    vocab,
    max_length=300,
)

# 5. Create training DataLoader.
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=partial(collate_fn, pad_id=vocab.pad_id),
)

# 6. Create validation DataLoader.
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=partial(collate_fn, pad_id=vocab.pad_id),
)

In [15]:
# Count positive and negative examples for each label.
positive_counts = train_df[LABEL_COLUMNS].sum().to_numpy()

negative_counts = (
    len(train_df) - positive_counts
)

# Calculate the imbalance ratio.
raw_weights = negative_counts / positive_counts

# Cap extreme weights.
pos_weights = np.clip(
    raw_weights,
    a_min=None,
    a_max=MAX_POS_WEIGHT,
)

# Convert weights into a PyTorch tensor.
pos_weight_tensor = torch.tensor(
    pos_weights,
    dtype=torch.float32,
    device=device,
)

print("Positive class weight analysis\n")

for label, positives, negatives, raw, final in zip(
    LABEL_COLUMNS,
    positive_counts,
    negative_counts,
    raw_weights,
    pos_weights,
):
    print(
        f"{label:15s} | "
        f"Positive: {positives:6.0f} | "
        f"Negative: {negatives:6.0f} | "
        f"Raw: {raw:8.2f} | "
        f"Applied: {final:5.2f}"
    )

Positive class weight analysis

toxic           | Positive:  12238 | Negative: 115418 | Raw:     9.43 | Applied:  9.43
severe_toxic    | Positive:   1274 | Negative: 126382 | Raw:    99.20 | Applied: 20.00
obscene         | Positive:   6734 | Negative: 120922 | Raw:    17.96 | Applied: 17.96
threat          | Positive:    404 | Negative: 127252 | Raw:   314.98 | Applied: 20.00
insult          | Positive:   6263 | Negative: 121393 | Raw:    19.38 | Applied: 19.38
identity_hate   | Positive:   1111 | Negative: 126545 | Raw:   113.90 | Applied: 20.00


In [18]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Fresh model
model = ToxicCommentBiLSTM(
    vocab_size=len(vocab),
    embedding_dim=100,
    hidden_size=128,
    num_labels=6,
    dropout=0.3,
).to(device)

# Same weighted loss
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)

# Fresh optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Device: cuda
Parameters: 3,237,062


In [19]:
checkpoint_path = Path("models/bilstm_ex2_best.pt")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

best_val_loss = float("inf")
best_epoch = None

history = {
    "train_loss": [],
    "val_loss": [],
    "epoch_time": [],
}

for epoch in range(EPOCHS):
    start_time = time.time()

    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
    )

    val_loss = evaluate(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=device,
    )

    epoch_time = time.time() - start_time

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["epoch_time"].append(epoch_time)

    # Save only when validation loss improves
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(
            {
                "epoch": best_epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": best_val_loss,
            },
            checkpoint_path,
        )

        checkpoint_status = "Best checkpoint saved"
    else:
        checkpoint_status = ""

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"| Train Loss: {train_loss:.6f} "
        f"| Val Loss: {val_loss:.6f} "
        f"| Time: {epoch_time:.2f}s "
        f"| {checkpoint_status}"
    )

print("\nTraining completed.")
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Checkpoint: {checkpoint_path}")

Epoch [1/5] | Train Loss: 0.389525 | Val Loss: 0.288217 | Time: 65.48s | Best checkpoint saved
Epoch [2/5] | Train Loss: 0.234610 | Val Loss: 0.243643 | Time: 65.31s | Best checkpoint saved
Epoch [3/5] | Train Loss: 0.181321 | Val Loss: 0.224035 | Time: 66.30s | Best checkpoint saved
Epoch [4/5] | Train Loss: 0.144707 | Val Loss: 0.252793 | Time: 66.70s | 
Epoch [5/5] | Train Loss: 0.117681 | Val Loss: 0.307338 | Time: 68.80s | 

Training completed.
Best epoch: 3
Best validation loss: 0.224035
Checkpoint: models/bilstm_ex2_best.pt


In [20]:
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded best checkpoint from epoch {checkpoint['epoch']}")
print(f"Best validation loss: {checkpoint['val_loss']:.6f}")

Loaded best checkpoint from epoch 3
Best validation loss: 0.224035


In [22]:
y_true, y_prob = predict(
    model=model,
    dataloader=val_loader,
    device=device,
)

results = calculate_metrics(
    y_true=y_true,
    y_prob=y_prob,
    threshold=0.5,
    label_names=LABEL_COLUMNS,
)

print("Per-label metrics:")
display(results["per_label"])

print("\nAggregate metrics:")
for metric, value in results["aggregate"].items():
    print(f"{metric}: {value:.4f}")

Per-label metrics:


,label,precision,recall,f1,support
0,toxic,0.567469,0.917866,0.701338,3056
1,severe_toxic,0.240233,0.900312,0.379265,321
2,obscene,0.530705,0.957434,0.682886,1715
3,threat,0.151420,0.648649,0.245524,74
4,insult,0.439702,0.951053,0.601371,1614
5,identity_hate,0.134355,0.775510,0.229031,294



Aggregate metrics:
micro_precision: 0.4440
micro_recall: 0.9255
micro_f1: 0.6001
macro_f1: 0.4732
weighted_f1: 0.6350


In [25]:
# Convert predictions to NumPy
y_true_np = y_true.cpu().numpy()
y_prob_np = y_prob.cpu().numpy()

thresholds = np.arange(0.05, 0.96, 0.05)

threshold_results = []

for i, label in enumerate(LABEL_COLUMNS):
    label_results = []

    for threshold in thresholds:
        predictions = (y_prob_np[:, i] >= threshold).astype(int)

        f1 = f1_score(
            y_true_np[:, i],
            predictions,
            zero_division=0,
        )

        label_results.append({
            "label": label,
            "threshold": round(threshold, 2),
            "f1": f1,
        })

    best_result = max(label_results, key=lambda x: x["f1"])

    threshold_results.extend(label_results)

    print(
        f"{label:15s} | "
        f"Best threshold: {best_result['threshold']:.2f} | "
        f"Best F1: {best_result['f1']:.4f}"
    )

threshold_df = pd.DataFrame(threshold_results)

toxic           | Best threshold: 0.90 | Best F1: 0.7972
severe_toxic    | Best threshold: 0.90 | Best F1: 0.5011
obscene         | Best threshold: 0.95 | Best F1: 0.7991
threat          | Best threshold: 0.80 | Best F1: 0.3420
insult          | Best threshold: 0.90 | Best F1: 0.7144
identity_hate   | Best threshold: 0.85 | Best F1: 0.3286


In [26]:
threshold_pivot = threshold_df.pivot(
    index="label",
    columns="threshold",
    values="f1",
)

display(threshold_pivot)

threshold,0.05,0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90,0.95
label,,,,,,,,,,,,,,,,,,,
identity_hate,0.107890,0.129381,0.145153,0.159714,0.173146,0.183688,0.193349,0.200501,0.212688,0.229031,0.242928,0.255556,0.265094,0.281911,0.304428,0.304933,0.328612,0.304094,0.189415
insult,0.376593,0.426340,0.459012,0.486319,0.510091,0.531229,0.550653,0.566575,0.583239,0.601371,0.614202,0.629514,0.644464,0.659632,0.675064,0.691292,0.707380,0.714366,0.701597
obscene,0.420870,0.492559,0.536655,0.569775,0.596178,0.618507,0.636242,0.652208,0.667476,0.682886,0.697883,0.713281,0.726207,0.741928,0.759149,0.771443,0.784445,0.796968,0.799061
severe_toxic,0.218081,0.254340,0.271768,0.292286,0.315026,0.328100,0.340766,0.358047,0.368454,0.379265,0.392588,0.407487,0.419940,0.420969,0.433613,0.449500,0.470238,0.501144,0.486331
threat,0.067079,0.094666,0.119843,0.137931,0.161702,0.183607,0.203704,0.222222,0.227790,0.245524,0.263305,0.287540,0.302158,0.322314,0.341232,0.341969,0.303030,0.283688,0.242991
toxic,0.437536,0.509932,0.555462,0.585371,0.608997,0.630212,0.650578,0.666355,0.683587,0.701338,0.713678,0.727249,0.743302,0.759157,0.772974,0.783549,0.792994,0.797170,0.784191
